# 01. Data Preparation & Cleaning
This notebook loads raw Citi Bike trip data, validates it, and prepares clean datasets for analysis


In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile
import json
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Setup paths
DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# 1.1 Data Loading

In [2]:
# Find all CSV ZIP files 
zip_files = sorted(RAW_DIR.glob('*.zip'))
print(f"Found {len(zip_files)} CSV ZIP files:")
for f in zip_files[:5]:  # Show first 5
    print(f"  - {f.name}")
if len(zip_files) > 5:
    print(f"  ... and {len(zip_files) - 5} more")


Found 6 CSV ZIP files:
  - 202401-citibike-tripdata.zip
  - 202402-citibike-tripdata.zip
  - 202403-citibike-tripdata.zip
  - 202404-citibike-tripdata.zip
  - 202405-citibike-tripdata.zip
  ... and 1 more


In [3]:
# Load and combine all CSVs
all_dfs = []
for zip_file in tqdm(zip_files, desc="Processing zip files"):
    with zipfile.ZipFile(zip_file) as z:
        for csv_name in z.namelist():
            if csv_name.endswith('.csv'):
                with z.open(csv_name) as f:
                    df = pd.read_csv(f)
                    all_dfs.append(df)

trips = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal trips loaded: {len(trips):,}")
print(f"Memory usage: {trips.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


Processing zip files: 100%|██████████| 6/6 [00:29<00:00,  5.00s/it]



Total trips loaded: 18,807,481
Memory usage: 10758.7 MB


In [4]:
# Inspect structure
print("\nDataset Info:")
print(trips.info())
print("\nFirst few rows:")
display(trips.head())



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18807481 entries, 0 to 18807480
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 1.8+ GB
None

First few rows:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,62EF1AC5BE598131,classic_bike,2024-01-24 09:03:33.533,2024-01-24 09:06:53.535,E 102 St & 1 Ave,7407.13,E 103 St & Lexington Ave,7463.09,40.786995,-73.941648,40.790305,-73.947558,member
1,8464E543DAB27DBF,classic_bike,2024-01-30 08:21:29.510,2024-01-30 08:29:03.304,E 102 St & 1 Ave,7407.13,E 91 St & 2 Ave,7286.01,40.786995,-73.941648,40.781153,-73.949630,member
2,9C04FDC8549F5205,electric_bike,2024-01-22 21:18:25.199,2024-01-22 21:26:24.647,W 35 St & 8 Ave,6526.01,1 Ave & E 39 St,6303.01,40.752762,-73.992805,40.747140,-73.971130,member
3,7DD1703A3E0D8833,electric_bike,2024-01-31 22:15:49.861,2024-01-31 22:22:45.520,Warren St & Roosevelt Ave,6346.07,112 St & Northern Blvd,6683.01,40.749190,-73.870540,40.757880,-73.857630,member
4,6A96FCD170996E59,classic_bike,2024-01-29 22:52:28.276,2024-01-29 22:57:05.099,6 Ave & W 33 St,6364.07,W 29 St & 9 Ave,6416.06,40.749013,-73.988484,40.750073,-73.998393,member


# 1.2 Data Quality Assessment

In [5]:
print("\n### Missing Values ###")
missing = trips.isnull().sum()
missing_pct = (missing / len(trips) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_pct
}).query('Missing > 0').sort_values('Missing', ascending=False)

print(missing_summary)


### Missing Values ###
                    Missing  Percentage
end_station_id        46485        0.25
end_lat               46312        0.25
end_lng               46312        0.25
end_station_name      43612        0.23
start_station_name    11770        0.06
start_station_id      11770        0.06
start_lat             11770        0.06
start_lng             11770        0.06


In [6]:
display(trips[trips['end_station_id'].isnull()].head())

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
141636,E62E28513EE08C6B,electric_bike,2024-01-19 07:12:00.109,2024-01-19 09:39:43.349,E 33 St & 1 Ave,6197.08,NaN,NaN,40.743227,-73.974498,NaN,NaN,member
141640,389546DFC23A23AE,electric_bike,2024-01-18 07:03:36.279,2024-01-18 08:04:48.735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,member
141641,985AE68FECBCEB82,electric_bike,2024-01-27 16:10:13.005,2024-01-27 17:10:23.567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,member
141642,BD69DB085E2E6556,electric_bike,2024-01-29 16:11:24.113,2024-01-29 17:12:24.979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,member
141643,7D54049196127961,electric_bike,2024-01-29 11:24:57.059,2024-01-29 12:37:24.432,Andrew Ave N & Hall of Fame Tce,8520.01,NaN,NaN,40.858340,-73.910080,NaN,NaN,casual


In 6 months of data, we will be able to retain ~99.7% of data

In [7]:
print("\n### Duplicate Rows ###")
duplicates = trips.duplicated().sum()
print(f"Duplicate rows: {duplicates:,} ({duplicates/len(trips)*100:.2f}%)")



### Duplicate Rows ###
Duplicate rows: 0 (0.00%)


# 1.3 Data Cleaning

In [8]:
trips_clean = trips.copy()

print("Starting cleaning process...")

Starting cleaning process...


In [9]:
# Step 1: Parse timestamps
print("\n[1/6] Parsing timestamps...")
trips_clean['started_at'] = pd.to_datetime(trips_clean['started_at'])
trips_clean['ended_at'] = pd.to_datetime(trips_clean['ended_at'])
print("✓ Timestamps parsed")


[1/6] Parsing timestamps...
✓ Timestamps parsed


In [10]:
# Step 3: Calc trip duration
print("\n[2/6] Calculating trip duration...")
trips_clean['trip_duration_minutes'] = (
    trips_clean['ended_at'] - trips_clean['started_at']
).dt.total_seconds() / 60
print("✓ Trip duration calculated")


[2/6] Calculating trip duration...
✓ Trip duration calculated


In [11]:
# Step 3: Handle missing values
print("\n[3/6] Handling missing values...")

# Drop rows with missing critical fields
critical_cols = [
    'started_at', 'ended_at', 
    'start_station_id', 'end_station_id',
    'start_lat', 'start_lng', 'end_lat', 'end_lng'
]

before_count = len(trips_clean)
trips_clean = trips_clean.dropna(subset=critical_cols)
after_count = len(trips_clean)

print(f"  Dropped {before_count - after_count:,} rows with missing critical fields")
print(f"  Remaining: {after_count:,} trips ({after_count/before_count*100:.1f}%)")



[3/6] Handling missing values...
  Dropped 51,732 rows with missing critical fields
  Remaining: 18,755,749 trips (99.7%)


In [12]:
# Step 4: Remove invalid trips
print("\n[4/6] Removing invalid trips...")

before_count = len(trips_clean)

# Remove trips with negative or zero duration
trips_clean = trips_clean[trips_clean['trip_duration_minutes'] > 0]

# Remove unrealistic durations (< 1 min or > 24 hours)
trips_clean = trips_clean[
    (trips_clean['trip_duration_minutes'] >= 1) & 
    (trips_clean['trip_duration_minutes'] <= 1440)
]

# Remove invalid coordinates (NYC area: lat 40.5-41.0, lng -74.5 to -73.5)
trips_clean = trips_clean[
    (trips_clean['start_lat'].between(40.5, 41.0)) &
    (trips_clean['start_lng'].between(-74.5, -73.5)) &
    (trips_clean['end_lat'].between(40.5, 41.0)) &
    (trips_clean['end_lng'].between(-74.5, -73.5))
]

after_count = len(trips_clean)
print(f"  Removed {before_count - after_count:,} invalid trips")
print(f"  Remaining: {after_count:,} trips ({after_count/before_count*100:.1f}%)")



[4/6] Removing invalid trips...
  Removed 220 invalid trips
  Remaining: 18,755,529 trips (100.0%)


In [13]:
# Step 5: Standardize station IDs and names
print("\n[5/6] Standardizing station data...")

# Convert station IDs to string (some datasets mix int/string)
trips_clean['start_station_id'] = trips_clean['start_station_id'].astype(str)
trips_clean['end_station_id'] = trips_clean['end_station_id'].astype(str)

# Strip whitespace from station names
if 'start_station_name' in trips_clean.columns:
    trips_clean['start_station_name'] = trips_clean['start_station_name'].str.strip()
if 'end_station_name' in trips_clean.columns:
    trips_clean['end_station_name'] = trips_clean['end_station_name'].str.strip()

print("✓ Station data standardized")



[5/6] Standardizing station data...
✓ Station data standardized


In [14]:
# Step 6: Add datetime features
print("\n[6/6] Adding datetime features...")

# Make sure only 2024 data is included (since we have some 2023 data)
trips_clean = trips_clean[trips_clean['started_at'] >= '2024-01-01']

trips_clean['year'] = trips_clean['started_at'].dt.year
trips_clean['month'] = trips_clean['started_at'].dt.month
trips_clean['day'] = trips_clean['started_at'].dt.day
trips_clean['hour'] = trips_clean['started_at'].dt.hour
trips_clean['dayofweek'] = trips_clean['started_at'].dt.dayofweek  # 0=Monday
trips_clean['is_weekend'] = trips_clean['dayofweek'].isin([5, 6]).astype(int)
trips_clean['date'] = trips_clean['started_at'].dt.date

print("✓ Datetime features added")



[6/6] Adding datetime features...
✓ Datetime features added


In [15]:
print("\n### Cleaning Summary ###")
print(f"Original trips: {len(trips):,}")
print(f"Clean trips: {len(trips_clean):,}")
print(f"Removed: {len(trips) - len(trips_clean):,} ({(len(trips) - len(trips_clean))/len(trips)*100:.2f}%)")



### Cleaning Summary ###
Original trips: 18,807,481
Clean trips: 18,755,165
Removed: 52,316 (0.28%)


# 1.4 Station Metadata Extraction

In [16]:
# Get unique start stations
start_stations = trips_clean.groupby('start_station_id').agg({
    'start_station_name': 'first',
    'start_lat': 'mean',
    'start_lng': 'mean',
    'ride_id': 'count'
}).rename(columns={
    'start_station_name': 'station_name',
    'start_lat': 'latitude',
    'start_lng': 'longitude',
    'ride_id': 'trip_count'
}).reset_index().rename(columns={'start_station_id': 'station_id'})

# Get unique end stations
end_stations = trips_clean.groupby('end_station_id').agg({
    'end_station_name': 'first',
    'end_lat': 'mean',
    'end_lng': 'mean',
    'ride_id': 'count'
}).rename(columns={
    'end_station_name': 'station_name',
    'end_lat': 'latitude',
    'end_lng': 'longitude',
    'ride_id': 'trip_count'
}).reset_index().rename(columns={'end_station_id': 'station_id'})

# Combine and deduplicate
all_stations = pd.concat([
    start_stations[['station_id', 'station_name', 'latitude', 'longitude']],
    end_stations[['station_id', 'station_name', 'latitude', 'longitude']]
]).drop_duplicates(subset='station_id').reset_index(drop=True)

print("\nUnique Start Stations: {:,}".format(len(start_stations)))
print("\nUnique End Stations: {:,}".format(len(end_stations)))
print(f"\nUnique stations Overall: {len(all_stations):,}")



Unique Start Stations: 2,336

Unique End Stations: 2,355

Unique stations Overall: 2,361


In [17]:
# Rename index properly
stations = trips_clean.groupby('start_station_id').agg({
    'start_station_name': 'first',
    'start_lat': 'mean',
    'start_lng': 'mean'
}).rename(columns={
    'start_station_name': 'station_name',
    'start_lat': 'latitude',
    'start_lng': 'longitude'
})

# Add trip counts
stations['departures'] = trips_clean.groupby('start_station_id').size()
stations['arrivals'] = trips_clean.groupby('end_station_id').size()
stations['total_trips'] = stations['departures'] + stations['arrivals']

stations = stations.reset_index().rename(columns={'start_station_id': 'station_id'})

print(f"Unique stations: {len(stations):,}")
print(f"\nTop 10 start stations by total trips:")
print(stations.nlargest(10, 'total_trips')[['station_id', 'station_name', 'total_trips']])


Unique stations: 2,336

Top 10 start stations by total trips:
     station_id             station_name  total_trips
1120    6140.05          W 21 St & 6 Ave     147792.0
1019    5905.14  University Pl & E 14 St     125868.0
1303    6450.05          8 Ave & W 31 St     121521.0
793     5329.03    West St & Chambers St     119366.0
1138    6173.08       Broadway & W 25 St     111765.0
1503    6822.09          1 Ave & E 68 St     111199.0
1243    6331.01          W 31 St & 7 Ave     106966.0
985     5788.13    Lafayette St & E 8 St     106570.0
1452    6726.01         11 Ave & W 41 St     106268.0
1154    6197.08          E 33 St & 1 Ave     104356.0


# 1.5 Create Hourly Demand Dataset

In [18]:
print("\n### Creating Hourly Demand Dataset ###")

# Aggregate departures by station and hour
hourly_demand = trips_clean.groupby([
    'start_station_id',
    pd.Grouper(key='started_at', freq='h')
]).size().reset_index(name='demand')

hourly_demand = hourly_demand.rename(columns={
    'start_station_id': 'station_id',
    'started_at': 'timestamp'
})

print(f"Hourly demand records: {len(hourly_demand):,}")
print(f"Date range: {hourly_demand['timestamp'].min()} to {hourly_demand['timestamp'].max()}")
print(f"Unique stations: {hourly_demand['station_id'].nunique()}")



### Creating Hourly Demand Dataset ###
Hourly demand records: 4,504,133
Date range: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
Unique stations: 2336


In [19]:
# Check coverage
print("\n### Data Coverage Analysis ###")

total_hours = (hourly_demand['timestamp'].max() - hourly_demand['timestamp'].min()).total_seconds() / 3600
expected_records = len(stations) * total_hours
actual_records = len(hourly_demand)

print(f"Expected records (all stations × all hours): {expected_records:,.0f}")
print(f"Actual records: {actual_records:,}")
print(f"Coverage: {actual_records/expected_records*100:.1f}%")
print("\nNote: Missing records indicate hours with zero demand")



### Data Coverage Analysis ###
Expected records (all stations × all hours): 10,201,312
Actual records: 4,504,133
Coverage: 44.2%

Note: Missing records indicate hours with zero demand


# 1.6 Save Processed Files

In [ ]:
print("\n### Saving Processed Data ###")

# Save trips
trips_path = PROCESSED_DIR / 'trips_clean.parquet'
trips_clean.to_parquet(trips_path, index=False)
print(f"✓ Saved trips: {trips_path}")
print(f"  Size: {trips_path.stat().st_size / 1024**2:.1f} MB")

# Save stations
stations_path = PROCESSED_DIR / 'stations_metadata.parquet'
stations.to_parquet(stations_path, index=False)
print(f"✓ Saved stations: {stations_path}")
print(f"  Size: {stations_path.stat().st_size / 1024:.1f} KB")

# Save hourly demand
demand_path = PROCESSED_DIR / 'hourly_demand.parquet'
hourly_demand.to_parquet(demand_path, index=False)
print(f"✓ Saved hourly demand: {demand_path}")
print(f"  Size: {demand_path.stat().st_size / 1024**2:.1f} MB")

# Save summary statistics
summary = {
    'data_preparation_date': datetime.now().isoformat(),
    'total_trips': int(len(trips_clean)),
    'unique_stations': int(len(stations)),
    'date_range': {
        'start': trips_clean['started_at'].min().isoformat(),
        'end': trips_clean['started_at'].max().isoformat()
    },
    'trip_duration_stats': {
        'mean_minutes': float(trips_clean['trip_duration_minutes'].mean()),
        'median_minutes': float(trips_clean['trip_duration_minutes'].median()),
        'std_minutes': float(trips_clean['trip_duration_minutes'].std())
    },
    'cleaning_summary': {
        'original_trips': int(len(trips)),
        'removed_trips': int(len(trips) - len(trips_clean)),
        'removal_rate_pct': float((len(trips) - len(trips_clean))/len(trips)*100)
    }
}

summary_path = PROCESSED_DIR / 'data_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
    
print(f"✓ Saved summary: {summary_path}")


print("\n" + "="*60)
print("DATA PREPARATION COMPLETE")
print("="*60)
print(f"\nProcessed {len(trips_clean):,} trips from {len(stations):,} stations")
print(f"Date range: {trips_clean['started_at'].min().date()} to {trips_clean['started_at'].max().date()}")


### Saving Processed Data ###
✓ Saved trips: ../data/processed/trips_clean.parquet
  Size: 988.3 MB
✓ Saved stations: ../data/processed/stations_metadata.parquet
  Size: 124.6 KB
✓ Saved hourly demand: ../data/processed/hourly_demand.parquet
  Size: 10.1 MB
✓ Saved summary: ../data/processed/data_summary.json

DATA PREPARATION COMPLETE

Processed 18,755,165 trips from 2,336 stations
Date range: 2024-01-01 to 2024-06-30
